In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

env= 'local'
env= 'colab'
env= 'kaggle'

# 核心参数
quick_test = False
random_seed = 42
micro_normalize = 'rank' # 'normal' 或 'rank'
best_k = 3
GLOBAL_DIST_TYPE = 't'

initial_train_years=15
val_years=1
test_years=1

epochs=500
batch_size=1024
lr=1e-3
patience=25

In [2]:

if env == 'kaggle':
    # This Python 3 environment comes with many helpful analytics libraries installed
    # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
    # For example, here's several helpful packages to load

    # Input data files are available in the read-only "../input/" directory
    # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

    micro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/micro_data.parquet')
    macro_df = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/macro_data.parquet')
    # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
    # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

elif env == 'colab':
    # Colab 版本的代码
    # 1. 设置环境变量
    import google.colab 
    os.environ['KAGGLE_USERNAME'] = 'jianbinchenuc'
    os.environ['KAGGLE_KEY'] = 'fa590c922d3624b8e0734dc7a90dd95a'

    # 2. 安装 kaggle 库 (如果尚未安装)
    !pip install -q kaggle

    # 3. 创建目录并下载数据集
    # -p 参数指定下载目录，--unzip 参数会自动解压
    !mkdir -p /content/mdndata
    !kaggle datasets download -d jianbinchenuc/mdndata-1990 -p /content/mdndata --unzip

    # 4. 验证文件是否已存在
    print("已下载的文件：", os.listdir('/content/mdndata'))
    micro_df = pd.read_parquet('/content/mdndata/micro_data.parquet') # 可以根据需要切换到 Colab 的路径
    macro_df = pd.read_parquet('/content/mdndata/macro_data.parquet') # 可以根据需要切换到 Colab 的路径

elif env == 'local':
    micro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/micro_data.parquet') # local 路径
    macro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/macro_data.parquet') # local 路径

else:
    raise ValueError("Invalid environment specified. Please choose 'kaggle', 'colab', or 'local'.")

Dataset URL: https://www.kaggle.com/datasets/jianbinchenuc/mdndata-1990
License(s): unknown
100% 226M/226M [00:14<00:00, 16.3MB/s] 

已下载的文件： ['macro_data.parquet', 'micro_data.parquet']


In [3]:
micro_df.head(5)

,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,ffi49,bm,evm,pcf,divyield,gprof,npm,accrual,cfm,at_turn,rd_sale,inv_turn,debt_at,intcov_ratio,short_debt,curr_ratio,cash_ratio,ps,opmad,de_ratio,mktcap,capei,debt_ebitda,ocf_lct,gpm,roe,linkdt,linkenddt
0,10001,1990-01-31,-0.018519,9.9375,10156.13,35255.0,1022.0,4920.0,012994,1990-01-31,31.0,0.917331,4.853678,7.769246,0.050633,0.164538,0.050236,-0.005309,0.081263,1.223659,0.0,91.308370,0.426239,3.131479,0.107954,1.173138,0.271814,0.421441,0.103437,2.235379,10.092250,16.372132,2.590528,0.339098,0.134464,0.151031,1986-01-09,2017-08-31
1,10001,1990-02-28,-0.006289,9.8750,10092.25,14862.0,1022.0,4920.0,012994,1990-02-28,31.0,0.849363,4.626443,5.646409,0.050000,0.181426,0.052654,-0.024148,0.084097,1.301493,0.0,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.403697,0.107955,2.084969,10.220000,15.386932,2.279116,0.508677,0.139398,0.162685,1986-01-09,2017-08-31
2,10001,1990-03-31,0.012821,9.8750,10141.63,12727.0,1027.0,4920.0,012994,1990-03-31,31.0,0.849363,4.626443,5.744959,0.049383,0.181426,0.052654,-0.024148,0.084097,1.301493,0.0,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.410743,0.107955,2.084969,10.398375,15.655488,2.279116,0.508677,0.139398,0.162685,1986-01-09,2017-08-31
3,10001,1990-04-30,0.000000,9.8750,10141.63,16645.0,1027.0,4920.0,012994,1990-04-30,31.0,0.849363,4.626443,5.674033,0.050000,0.181426,0.052654,-0.024148,0.084097,1.301493,0.0,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.405672,0.107955,2.084969,10.270000,15.462210,2.279116,0.508677,0.139398,0.162685,1986-01-09,2017-08-31
4,10001,1990-05-31,-0.012658,9.7500,10013.25,27885.0,1027.0,4920.0,012994,1990-05-31,31.0,0.911008,4.819588,4.424768,0.051282,0.182316,0.056644,-0.047016,0.089212,1.223747,0.0,89.841695,0.403169,3.227166,0.066927,1.278309,0.263992,0.422964,0.116415,1.943177,10.013250,14.173036,2.211369,0.693056,0.148982,0.158487,1986-01-09,2017-08-31


In [4]:
micro_df['public_date'].isna().sum()

np.int64(129777)

In [5]:
micro_df.shape

(2226544, 38)

In [6]:
micro_df['mthret'].isna().sum()

np.int64(35466)

In [7]:
micro_df[micro_df['public_date'].isna()].head(5)

,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,ffi49,bm,evm,pcf,divyield,gprof,npm,accrual,cfm,at_turn,rd_sale,inv_turn,debt_at,intcov_ratio,short_debt,curr_ratio,cash_ratio,ps,opmad,de_ratio,mktcap,capei,debt_ebitda,ocf_lct,gpm,roe,linkdt,linkenddt
215,10001,2007-12-31,-0.004206,14.1400,40652.50,46981.0,2875.0,4920.0,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
216,10001,2008-01-31,-0.005890,14.0000,40250.00,68545.0,2875.0,4920.0,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
217,10001,2008-02-29,0.022010,9.4999,41277.07,42393.0,4345.0,4920.0,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
606,10002,2012-12-31,0.010946,2.7800,49945.48,304780.0,17966.0,6020.0,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
607,10002,2013-01-31,0.035971,2.8800,51742.08,303580.0,17966.0,6020.0,None,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT


In [5]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
import random
import time
from datetime import datetime
from scipy.stats import t      # 🌟 换成 t 分布
from scipy.optimize import brentq
from tqdm import tqdm
from scipy.stats import norm

warnings.filterwarnings('ignore')

# ==========================================
# 全局随机种子锁死函数
# ==========================================
def seed_everything(seed=42):
    """
    全局锁死随机种子，确保深度学习模型的绝对可复现性
    """
    # 1. 锁死 Python 内置随机库
    random.seed(seed)
    
    # 2. 锁死 Python 哈希表的随机化 (保证字典遍历顺序一致)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. 锁死 Numpy 随机种子
    np.random.seed(seed)
    
    # 4. 锁死 PyTorch CPU 随机种子
    torch.manual_seed(seed)
    
    # 5. 锁死 PyTorch GPU 随机种子
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # 如果你有双卡/多卡
        
        # 【极其关键】锁死 CuDNN 底层算法的随机性
        # 注意：设置为 True 可能会略微牺牲一点点 GPU 训练速度，但为了发论文复现，这是必须的！
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"Global seed set to {seed} to ensure reproducibility.")

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
# ==========================================
# 1. 外置数据预处理模块 (ETL & Feature Engineering)
# ==========================================
print(">>> [1/3] Loading and Preprocessing Data (Externalized)...")

# micro_cols 是微观特征列，macro_cols 是宏观特征列，后续我们会对它们分别进行不同的预处理
micro_cols=micro_df.columns.drop(['permno','mthcaldt', 'mthprc','shrout','siccd','gvkey', 'public_date','ffi49','linkdt'	,'linkenddt']).tolist()
macro_cols=macro_df.columns.drop(['date']).tolist()

# 1.1 日期对齐与合并
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df['date'] = pd.to_datetime(macro_df['date']) + pd.offsets.MonthEnd(0)

df = pd.merge(micro_df, macro_df, left_on='mthcaldt', right_on='date', how='left')
df = df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)
df.head(5)

>>> [1/3] Loading and Preprocessing Data (Externalized)...


,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,ffi49,bm,evm,pcf,divyield,gprof,npm,accrual,cfm,at_turn,rd_sale,inv_turn,debt_at,intcov_ratio,short_debt,curr_ratio,cash_ratio,ps,opmad,de_ratio,mktcap,capei,debt_ebitda,ocf_lct,gpm,roe,linkdt,linkenddt,date,ret,tms,dfy,dfr,svar,vrp,lzrt,wtexas,skvw,tail,dtoy,dtoat,rdsp,avgcor,d/p,d/y,infl,ntis,ndrbl,disag,e/p,d/e,ygap,b/m,i/k,govik,crdstd
0,10001,1990-01-31,-0.018519,9.9375,10156.13,35255.0,1022.0,4920.0,012994,1990-01-31,31.0,0.917331,4.853678,7.769246,0.050633,0.164538,0.050236,-0.005309,0.081263,1.223659,0.0,91.308370,0.426239,3.131479,0.107954,1.173138,0.271814,0.421441,0.103437,2.235379,10.092250,16.372132,2.590528,0.339098,0.134464,0.151031,1986-01-09,2017-08-31,1990-01-31,-0.067661,0.0101,0.0095,0.0152,0.002892,35.9054,-1.672983,0.040330,0.049831,0.465337,0.921851,0.921851,0.022034,0.307570,0.031282,0.031952,0.001589,-0.012334,0.065133,2.941912,0.067850,0.453103,-2.769167,0.399210,0.035913,0.034468,54.4
1,10001,1990-02-28,-0.006289,9.8750,10092.25,14862.0,1022.0,4920.0,012994,1990-02-28,31.0,0.849363,4.626443,5.646409,0.050000,0.181426,0.052654,-0.024148,0.084097,1.301493,0.0,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.403697,0.107955,2.084969,10.220000,15.386932,2.279116,0.508677,0.139398,0.162685,1986-01-09,2017-08-31,1990-02-28,0.013381,0.0102,0.0092,0.0013,0.001009,32.9177,-1.558446,-0.051101,0.058325,0.476048,0.934915,0.934915,0.018495,0.316860,0.033860,0.031530,0.010309,-0.013897,0.027459,2.929758,0.068800,0.462961,-2.753609,0.406415,0.035913,0.034468,54.4
2,10001,1990-03-31,0.012821,9.8750,10141.63,12727.0,1027.0,4920.0,012994,1990-03-31,31.0,0.849363,4.626443,5.744959,0.049383,0.181426,0.052654,-0.024148,0.084097,1.301493,0.0,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.410743,0.107955,2.084969,10.398375,15.655488,2.279116,0.508677,0.139398,0.162685,1986-01-09,2017-08-31,1990-03-31,0.026588,0.0099,0.0084,0.0033,0.001032,26.5978,-1.523794,-0.058496,0.009316,0.488723,0.963369,0.963369,0.023228,0.335480,0.033838,0.034126,0.004710,-0.011729,-0.003898,3.042521,0.066890,0.473052,-2.780460,0.397226,0.035913,0.034468,54.4
3,10001,1990-04-30,0.000000,9.8750,10141.63,16645.0,1027.0,4920.0,012994,1990-04-30,31.0,0.849363,4.626443,5.674033,0.050000,0.181426,0.052654,-0.024148,0.084097,1.301493,0.0,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.405672,0.107955,2.084969,10.270000,15.462210,2.279116,0.508677,0.139398,0.162685,1986-01-09,2017-08-31,1990-04-30,-0.024504,0.0147,0.0084,0.0011,0.000967,26.2753,-1.525570,-0.085306,0.007114,0.486923,0.945416,0.945416,0.023264,0.275801,0.033294,0.034102,0.005469,-0.010291,0.048815,3.071526,0.064714,0.483384,-2.813253,0.390455,0.035166,0.034878,54.4
4,10001,1990-05-31,-0.012658,9.7500,10013.25,27885.0,1027.0,4920.0,012994,1990-05-31,31.0,0.911008,4.819588,4.424768,0.051282,0.182316,0.056644,-0.047016,0.089212,1.223747,0.0,89.841695,0.403169,3.227166,0.066927,1.278309,0.263992,0.422964,0.116415,1.943177,10.013250,14.173036,2.211369,0.693056,0.148982,0.158487,1986-01-09,2017-08-31,1990-05-31,0.097419,0.0109,0.0094,-0.0030,0.001361,19.2201,-1.530473,-0.061995,0.045464,0.497122,0.999340,0.999340,0.028805,0.292450,0.034562,0.033632,0.001554,-0.010149,0.016941,2.916158,0.068281,0.495891,-2.763023,0.414971,0.035166,0.034878,54.4


In [10]:
df.shape

(2226544, 66)

In [11]:
df['permno'].isna().sum()

np.int64(0)

In [7]:

# =========================================================
# 在无缝网格上安全计算特征
# 此时，1 行 (row) 绝对等于 1 个月 (month)
# =========================================================

# 1. 计算 t+1 收益率 (如果是断层前的最后一个月，它会极其安全地抓取到 NaN)
df['target_ret_final'] = micro_df.groupby('permno')['mthret'].shift(-1)

# 2. 计算过去 12 个月动量 (跳过近1月，滚动11月)
df['log_ret'] = np.log1p(df['mthret'].fillna(0))

# 注意：因为中间补了 NaN，所以遇到断层时，滚动的 11 个月里只要有效月份少于 min_periods (8)，
# 动量计算结果就会自动变成 NaN，完美避免了跨越几年的荒谬计算！
df['MOM12m'] = df.groupby('permno')['log_ret'].transform(
    lambda x: x.shift(1).rolling(window=11, min_periods=8).sum()
)
df['MOM12m'] = np.exp(df['MOM12m']) - 1
#
print(" -> Reindexing and feature engineering complete!")
print(f" -> Total rows after reindexing: {len(df)}")
print(f" -> Total missing values in mthret_lead1 (should be > 0 due to gaps): {df['target_ret_final'].isna().sum()}")

# 3. 在完整的全局时间序列上，计算 12 个月滚动均值和标准差
df['mu_bench'] = df.groupby('permno')['mthret'].transform(lambda x: x.rolling(12, min_periods=3).mean())
df['sigma_bench'] = df.groupby('permno')['mthret'].transform(lambda x: x.rolling(12, min_periods=3).std())

# add new features to micro_cols
micro_cols = micro_cols+['MOM12m', 'mu_bench', 'sigma_bench']

 -> Reindexing and feature engineering complete!
 -> Total rows after reindexing: 2226544
 -> Total missing values in mthret_lead1 (should be > 0 due to gaps): 54028


In [13]:
df.head(5)

,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,ffi49,bm,evm,pcf,divyield,gprof,npm,accrual,cfm,at_turn,rd_sale,inv_turn,debt_at,intcov_ratio,short_debt,curr_ratio,cash_ratio,ps,opmad,de_ratio,mktcap,capei,debt_ebitda,ocf_lct,gpm,roe,linkdt,linkenddt,date,ret,tms,dfy,dfr,svar,vrp,lzrt,wtexas,skvw,tail,dtoy,dtoat,rdsp,avgcor,d/p,d/y,infl,ntis,ndrbl,disag,e/p,d/e,ygap,b/m,i/k,govik,crdstd,target_ret_final,log_ret,MOM12m,mu_bench,sigma_bench
0,10001,1990-01-31,-0.018519,9.9375,10156.13,35255.0,1022.0,4920.0,012994,1990-01-31,31.0,0.917331,4.853678,7.769246,0.050633,0.164538,0.050236,-0.005309,0.081263,1.223659,0.0,91.308370,0.426239,3.131479,0.107954,1.173138,0.271814,0.421441,0.103437,2.235379,10.092250,16.372132,2.590528,0.339098,0.134464,0.151031,1986-01-09,2017-08-31,1990-01-31,-0.067661,0.0101,0.0095,0.0152,0.002892,35.9054,-1.672983,0.040330,0.049831,0.465337,0.921851,0.921851,0.022034,0.307570,0.031282,0.031952,0.001589,-0.012334,0.065133,2.941912,0.067850,0.453103,-2.769167,0.399210,0.035913,0.034468,54.4,-0.006289,-0.018693,NaN,NaN,NaN
1,10001,1990-02-28,-0.006289,9.8750,10092.25,14862.0,1022.0,4920.0,012994,1990-02-28,31.0,0.849363,4.626443,5.646409,0.050000,0.181426,0.052654,-0.024148,0.084097,1.301493,0.0,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.403697,0.107955,2.084969,10.220000,15.386932,2.279116,0.508677,0.139398,0.162685,1986-01-09,2017-08-31,1990-02-28,0.013381,0.0102,0.0092,0.0013,0.001009,32.9177,-1.558446,-0.051101,0.058325,0.476048,0.934915,0.934915,0.018495,0.316860,0.033860,0.031530,0.010309,-0.013897,0.027459,2.929758,0.068800,0.462961,-2.753609,0.406415,0.035913,0.034468,54.4,0.012821,-0.006309,NaN,NaN,NaN
2,10001,1990-03-31,0.012821,9.8750,10141.63,12727.0,1027.0,4920.0,012994,1990-03-31,31.0,0.849363,4.626443,5.744959,0.049383,0.181426,0.052654,-0.024148,0.084097,1.301493,0.0,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.410743,0.107955,2.084969,10.398375,15.655488,2.279116,0.508677,0.139398,0.162685,1986-01-09,2017-08-31,1990-03-31,0.026588,0.0099,0.0084,0.0033,0.001032,26.5978,-1.523794,-0.058496,0.009316,0.488723,0.963369,0.963369,0.023228,0.335480,0.033838,0.034126,0.004710,-0.011729,-0.003898,3.042521,0.066890,0.473052,-2.780460,0.397226,0.035913,0.034468,54.4,0.000000,0.012740,NaN,-0.003996,0.015795
3,10001,1990-04-30,0.000000,9.8750,10141.63,16645.0,1027.0,4920.0,012994,1990-04-30,31.0,0.849363,4.626443,5.674033,0.050000,0.181426,0.052654,-0.024148,0.084097,1.301493,0.0,96.509413,0.413490,3.119863,0.088959,1.243238,0.273871,0.405672,0.107955,2.084969,10.270000,15.462210,2.279116,0.508677,0.139398,0.162685,1986-01-09,2017-08-31,1990-04-30,-0.024504,0.0147,0.0084,0.0011,0.000967,26.2753,-1.525570,-0.085306,0.007114,0.486923,0.945416,0.945416,0.023264,0.275801,0.033294,0.034102,0.005469,-0.010291,0.048815,3.071526,0.064714,0.483384,-2.813253,0.390455,0.035166,0.034878,54.4,-0.012658,0.000000,NaN,-0.002997,0.013051
4,10001,1990-05-31,-0.012658,9.7500,10013.25,27885.0,1027.0,4920.0,012994,1990-05-31,31.0,0.911008,4.819588,4.424768,0.051282,0.182316,0.056644,-0.047016,0.089212,1.223747,0.0,89.841695,0.403169,3.227166,0.066927,1.278309,0.263992,0.422964,0.116415,1.943177,10.013250,14.173036,2.211369,0.693056,0.148982,0.158487,1986-01-09,2017-08-31,1990-05-31,0.097419,0.0109,0.0094,-0.0030,0.001361,19.2201,-1.530473,-0.061995,0.045464,0.497122,0.999340,0.999340,0.028805,0.292450,0.034562,0.033632,0.001554,-0.010149,0.016941,2.916158,0.068281,0.495891,-2.763023,0.414971,0.035166,0.034878,54.4,0.013750,-0.012739,NaN,-0.004929,0.012100


In [8]:
# =========================================================
# 清洗没用的行
# =========================================================
# 每清洗一步都打印一下当前的行数和缺失值统计，确保我们没有过度清洗
print(">>> [2/3] Cleaning Data...")
# 把ffi49缺失的行变成0, 把这一列的类型改成整数（因为它是行业分类，缺失就当做0类）
df['ffi49'] = df['ffi49'].fillna(0).astype(int)
print(f"Initial rows: {len(df)}")
# 清洗mthret_lead1缺失的行（这些行无法用于训练，因为没有标签了）
# 更加语义化、高效的写法（底层经过 C 语言优化，比布尔索引更快）
df = df.dropna(subset=['target_ret_final', 'public_date']).reset_index(drop=True)
# df = df[~df['target_ret_final'].isna()].reset_index(drop=True)
print(f"After dropping rows with missing mthret_lead1 and public_date: {len(df)}")
# 清洗mthprc小于5的行（这些行可能是数据错误或者极端小盘股，训练时可能会引入噪音）
df = df[df['mthprc'] >= 5].reset_index(drop=True)
print(f"After dropping rows with mthprc < 5: {len(df)}")
# 清洗public_date为空的行（这些行可能是数据错误，训练时可能会引入噪音）
# df = df[df['public_date'].notnull()].reset_index(drop=True)
# print(f"After dropping rows with missing public_date: {len(df)}")
# 清洗mthcaldt在指定范围外的行
df = df[ (df['mthcaldt'] >= pd.to_datetime('1991-01-31')) & (df['mthcaldt'] <= pd.to_datetime('2023-12-31')) ]  # 日期在1990-01-31到2023-12-31之间
print(f"After filtering by date: {len(df)}")

>>> [2/3] Cleaning Data...
Initial rows: 2226544
After dropping rows with missing mthret_lead1 and public_date: 2084584
After dropping rows with mthprc < 5: 1562411
After filtering by date: 1461459


In [9]:
# ==========================================
# 1. 外置数据预处理模块 (ETL & Feature Engineering)
# ==========================================
# print(">>> [1/3] Loading and Preprocessing Data (Externalized)...")

# # micro_cols 是微观特征列，macro_cols 是宏观特征列，后续我们会对它们分别进行不同的预处理
# micro_cols=micro_df.columns.drop(['permno','mthcaldt', 'mthprc','siccd','gvkey', 'public_date','ff49','linkdt'	,'linkenddt'])
# macro_cols=macro_df.columns.drop(['date'])

# # 1.1 日期对齐与合并
# micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
# macro_df['date'] = pd.to_datetime(macro_df['date']) + pd.offsets.MonthEnd(0)

# df = pd.merge(micro_df, macro_df, left_on='mthcaldt', right_on='date', how='left')
# df = df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)



# df = df[ (df['mthcaldt'] >= pd.to_datetime('1990-01-31')) & (df['mthcaldt'] <= pd.to_datetime('2023-12-31')) ]  # 日期在1990-01-31到2023-12-31之间

# # 1.2 目标收益率设定 (绝对禁止缩尾！)
# df['target_ret_final'] = df['mthret_lead1']

# 1.3 缺失值预填充 (可在此处放置你的业务填充逻辑) 
# 例如：使用截面均值填充微观特征缺失值
for feature in micro_cols:
    if feature in df.columns:
        df[feature] = df[feature].fillna(df.groupby('mthcaldt')[feature].transform('mean'))

# 1.4 微观特征：截面秩标准化 (Cross-Sectional Rank Normalization) [-1, 1], 这个未来也可以考虑别的归一化的方法
print(" -> Applying Rank Normalization to Micro Features...")
micro_tensor_cols = []

# Micro 特征的归一化方法选择：秩标准化（Rank Normalization）或 Z-score 标准化
if micro_normalize == 'rank':
    # rank transformation
    for col in micro_cols:
        norm_col = f'{col}_norm'
        # 使用 pct=True 转化为 0~1，再映射到 -1~1
        df[norm_col] = df.groupby('mthcaldt')[col].transform(
            lambda x: (x.rank(pct=True) * 2) - 1
        ).fillna(0)  # 如果全截面缺失，填0(中性)
        micro_tensor_cols.append(norm_col)
else:
    # Z-score normalization
    for col in micro_cols:
        norm_col = f'{col}_norm'
        df[norm_col] = df.groupby('mthcaldt')[col].transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-8)
        ).fillna(0)  # 如果全截面缺失，填0(中性)
        micro_tensor_cols.append(norm_col)

# 1.5 宏观特征：Expanding Window 标准化 (Expanding Window Standardization)
# 在下面Expanding Window 逐窗口计算均值和标准差，保证每个月的宏观特征只使用当月及之前的数据进行标准化，完全避免未来数据泄漏
macro_tensor_cols = list(macro_cols)

# 1.6 清理最终的空值行并固化数据集
print(f" -> Finalizing dataset by dropping rows with missing target or features...")
print(f"Initial shape before dropping NA: {df.shape}")
# processed_df = df.dropna(subset=['target_ret_final'] + micro_tensor_cols + macro_tensor_cols).copy()
processed_df = df.copy()
print(f"Data preprocessing complete! Final shape: {processed_df.shape}")
print(f"Micro Features: {len(micro_tensor_cols)}, Macro Features: {len(macro_tensor_cols)}")

pd.concat([processed_df.iloc[:5], processed_df.iloc[-5:]])

 -> Applying Rank Normalization to Micro Features...
 -> Finalizing dataset by dropping rows with missing target or features...
Initial shape before dropping NA: (1461459, 102)
Data preprocessing complete! Final shape: (1461459, 102)
Micro Features: 31, Macro Features: 27


,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,ffi49,bm,evm,pcf,divyield,gprof,npm,accrual,cfm,at_turn,rd_sale,inv_turn,debt_at,intcov_ratio,short_debt,curr_ratio,cash_ratio,ps,opmad,de_ratio,mktcap,capei,debt_ebitda,ocf_lct,gpm,roe,linkdt,linkenddt,date,ret,tms,dfy,dfr,svar,vrp,lzrt,wtexas,skvw,tail,dtoy,dtoat,rdsp,avgcor,d/p,d/y,infl,ntis,ndrbl,disag,e/p,d/e,ygap,b/m,i/k,govik,crdstd,target_ret_final,log_ret,MOM12m,mu_bench,sigma_bench,mthret_norm,mthcap_norm,mthvol_norm,bm_norm,evm_norm,pcf_norm,divyield_norm,gprof_norm,npm_norm,accrual_norm,cfm_norm,at_turn_norm,rd_sale_norm,inv_turn_norm,debt_at_norm,intcov_ratio_norm,short_debt_norm,curr_ratio_norm,cash_ratio_norm,ps_norm,opmad_norm,de_ratio_norm,mktcap_norm,capei_norm,debt_ebitda_norm,ocf_lct_norm,gpm_norm,roe_norm,MOM12m_norm,mu_bench_norm,sigma_bench_norm
12,10001,1991-01-31,0.013158,9.625,1.014475e+04,3.839800e+04,1054.0,4920.0,012994,1991-01-31,31,0.896385,5.690808,4.872598,0.057143,0.161306,0.044401,-0.055775,0.082956,1.166318,0.000000,87.798432,0.392076,2.770452,0.061202,1.340247,0.316439,0.445982,0.099749,1.796365,10.14475,13.581666,2.430626,0.672101,0.138304,0.112235,1986-01-09,2017-08-31,1991-01-31,0.045001,0.0215,0.0141,0.002000,0.003194,19.8734,-1.605180,-0.232068,0.108935,0.496429,0.912206,0.912206,0.029781,0.311684,0.036600,0.037509,0.000000,-0.001846,0.031747,2.762114,0.071034,0.544342,-2.729763,0.520290,0.034366,0.035092,46.7,0.012987,0.013072,0.009425,0.002110,0.022682,-0.348973,-0.927397,-0.686301,-0.026027,-0.500000,-0.303425,0.709589,-0.436301,0.018493,-0.358219,0.063699,-0.039041,-0.324315,0.950000,0.636301,-0.400000,-0.587671,-0.619863,-0.152055,-0.441096,0.113699,0.181507,-0.924658,-0.171918,0.140411,0.520548,-0.561644,-0.106164,0.254110,-0.074658,-0.989726
13,10001,1991-02-28,0.012987,9.750,1.027650e+04,2.452500e+04,1054.0,4920.0,012994,1991-02-28,31,0.975332,5.299653,5.205927,0.056410,0.168351,0.047361,-0.042869,0.085157,1.192696,0.000000,89.490628,0.378677,3.063990,0.050190,1.348320,0.289359,0.434947,0.103356,1.755852,10.27650,13.406585,2.249325,0.614188,0.141152,0.121406,1986-01-09,2017-08-31,1991-02-28,0.071565,0.0247,0.0124,0.009100,0.002431,24.7257,-1.748574,-0.124542,0.059304,0.461125,0.960807,0.960807,0.032798,0.293192,0.035164,0.036624,0.005979,0.000722,-0.000872,2.781227,0.071075,0.551589,-2.727631,0.522452,0.034366,0.035092,46.7,-0.012244,0.012903,0.029179,0.003717,0.022716,-0.605561,-0.932105,-0.797608,0.066925,-0.291303,-0.249273,0.759457,-0.416747,0.116715,-0.165212,0.123181,0.207242,-0.325251,0.956030,0.584869,-0.304882,-0.629486,-0.599095,-0.169738,-0.482056,0.211122,0.180084,-0.926285,-0.224701,0.328160,0.493695,-0.770449,-0.001617,0.223408,-0.205949,-0.991594
14,10001,1991-03-31,-0.012244,9.500,1.001300e+04,1.243900e+04,1054.0,4920.0,012994,1991-03-31,31,0.975332,5.299653,5.072442,0.057895,0.168351,0.047361,-0.042869,0.085157,1.192696,0.000000,89.490628,0.378677,3.063990,0.050190,1.348320,0.289359,0.423795,0.103356,1.755852,10.01300,13.062826,2.249325,0.614188,0.141152,0.121406,1986-01-09,2017-08-31,1991-03-31,0.024354,0.0253,0.0116,0.007000,0.001375,14.9701,-1.639037,0.027197,0.013972,0.465335,0.971368,0.971368,0.032859,0.290643,0.032969,0.035187,0.001486,0.001247,0.010722,2.829259,0.066642,0.558926,-2.788988,0.498506,0.034366,0.035092,46.7,0.039474,-0.012320,0.029348,0.001628,0.022954,-0.543055,-0.937146,-0.869893,0.090509,-0.521684,-0.291012,0.797612,-0.410434,0.126964,-0.152106,0.142049,0.211188,-0.324953,0.956003,0.577624,-0.290383,-0.630421,-0.597109,-0.169076,-0.530484,0.215588,0.177876,-0.932747,-0.285984,0.328724,0.490258,-0.758642,0.245757,0.019485,-0.331238,-0.992458
15,10001,1991-04-30,0.039474,9.875,1.040825e+04,2.434300e+04,1054.0,4920.0,012994,1991-04-30,31,0.975332,5.299653,5.339412,0.055000,0.168351,0.047361,-0.042869,0.085157,1.192696,0.000000,89.490628,0.378677,3.063990,0.050190,1.348320,0.289359,0.446100,0.103356,1.755852,10.54000,13.750343,2.249325,0.614188,0.141152,0.121406,1986-

In [10]:
# ==========================================
# 2. 轻量级时间窗口生成器
# ==========================================
class DataWindowGenerator:
    """
    仅负责按时间切分数据，向模型输送 Train/Val/Test 窗口
    """
    def __init__(self, df, date_col='mthcaldt'):
        self.df = df
        self.date_col = date_col
        
    def expanding_window_generator(self, initial_train_years=20, val_years=2, test_years=1):
        dates = np.sort(self.df[self.date_col].unique())
        initial_train_months = initial_train_years * 12
        val_months = val_years * 12
        test_months = test_years * 12
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months < len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months

# ==========================================
# 2. 模型定义：结构化 MDN (Mixture Density Network)
# ==========================================
class StructuredMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, micro_hidden_dim=128, macro_hidden_dim=64, num_components=3, dropout_rate=0.2, dist_type='normal'):
        """
        升级版结构化 MDN：
        - micro_hidden_dim 提升至 128 (为未来扩展到 100+ 特征做准备)
        - macro_hidden_dim 提升至 64 (为未来扩展到 100+ 特征做准备)
        - 加入 Dropout 防止对某个单一因子过度依赖 (过拟合)
        """
        super(StructuredMDN, self).__init__()
        self.num_components = num_components
        self.dist_type = dist_type
        
        # ==========================================
        # 通道 A: Macro Network (门控网络 -> 预测机制权重 pi)
        # 宏观数据通常维度低且稳定，不需要太深
        # ==========================================
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, macro_hidden_dim),
            nn.LayerNorm(macro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate), # 随机屏蔽一部分宏观因子，防止死记硬背
            nn.Linear(macro_hidden_dim , num_components)
        )
        
        # ==========================================
        # 通道 B: Micro Network (专家网络 -> 预测 mu 和 sigma)
        # 微观数据维度高、噪音大，稍微加深一点，并强力正则化
        # ==========================================
        self.micro_extractor = nn.Sequential(
            nn.Linear(micro_dim, micro_hidden_dim),
            nn.BatchNorm1d(micro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(micro_hidden_dim, micro_hidden_dim // 2),
            nn.BatchNorm1d(micro_hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate)
        )
        
        # 预测头 (Heads)
        self.mu_head = nn.Linear(micro_hidden_dim // 2, num_components)
        
        # Sigma 必须严格为正，使用 Softplus
        self.sigma_head = nn.Linear(micro_hidden_dim // 2, num_components)

        # 🌟 仅在选择了 t 分布时，才挂载自由度预测头
        nu = None # 默认为空
        if dist_type == 't':
            self.nu_head = nn.Linear(micro_hidden_dim // 2, num_components)

    def forward(self, x_macro, x_micro):
        pi_logits = self.macro_net(x_macro)
        h_micro = self.micro_extractor(x_micro)
        
        mu = self.mu_head(h_micro)
        sigma = nn.functional.softplus(self.sigma_head(h_micro)) + 1e-6 # softplus=log(1+exp(x)), 可以确保输出严格为正，避免数值不稳定
        
        if self.dist_type == 't':
            nu = nn.functional.softplus(self.nu_head(h_micro)) + 1e-6

        return pi_logits, mu, sigma, nu

    
# ==========================================
# 2. 模型定义：结构化 MDN (Mixture Density Network) + FiLM 调制
# ==========================================
class StructuredMDN_with_FiLM(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, ind_emb_dim=6, 
                 micro_hidden_dim=128, macro_hidden_dim=64, num_components=3, 
                 dropout_rate=0.2, dist_type='normal'):
        """
        结构化 MDN (加入 FiLM 调制机制)
        - 保持 2D 截面输入，不引入序列记忆 (无 GRU)
        - 宏观特征不仅预测 Regime 概率，还直接调控微观特征的表达
        """
        super(StructuredMDN_with_FiLM, self).__init__()
        self.num_components = num_components
        self.dist_type = dist_type

        # ++++++++++++++++++++++++++++++++++++++++++
        # 新增：行业实体嵌入层 (Entity Embedding)
        # ++++++++++++++++++++++++++++++++++++++++++
        self.ind_embedding = nn.Embedding(num_embeddings=num_industries, embedding_dim=ind_emb_dim)
        
        # 计算拼接后的微观特征总维度 = 连续特征维度 + 行业嵌入维度(6)
        total_micro_dim = micro_dim + ind_emb_dim
        # ==========================================
        # 通道 A: Macro Network (标准 MLP)
        # 输入: (Batch, Macro_Dim) 截面宏观数据
        # ==========================================
        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, macro_hidden_dim),
            nn.LayerNorm(macro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(macro_hidden_dim , macro_hidden_dim//2), # 🌟 优化：增加一层，让宏观特征更丰富
            nn.LayerNorm(macro_hidden_dim//2),
            nn.ELU()
        )
        
        # 门控预测头：基于宏观隐状态预测机制概率 pi
        self.pi_head = nn.Linear(macro_hidden_dim//2, num_components)
        
        # ==========================================
        # ★ FiLM 调制发生器 (Modulation Generators) ★
        # ==========================================
        # 目标是对齐微观网络的最后一层输出维度
        target_dim = 16
        
        # Gamma 负责“缩放(Scale)”微观特征，Beta 负责“平移(Shift)”微观特征
        self.film_gamma = nn.Linear(macro_hidden_dim//2, target_dim)
        self.film_beta = nn.Linear(macro_hidden_dim//2, target_dim)
        
        # ==========================================
        # 通道 B: Micro Network (专家网络)
        # 输入: (Batch, Micro_Dim) 截面微观数据
        # ==========================================
        self.micro_extractor = nn.Sequential(
            nn.Linear(total_micro_dim, micro_hidden_dim),
            nn.LayerNorm(micro_hidden_dim), # 使用 LayerNorm 防止截面前视偏差
            nn.ELU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(micro_hidden_dim, micro_hidden_dim // 2),
            nn.LayerNorm(micro_hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(micro_hidden_dim // 2, target_dim),
            nn.LayerNorm(target_dim),
            nn.ELU()
        )
        
        # 预测头 (Heads): 注意，这里输入的是被 FiLM 调制后的特征
        self.mu_head = nn.Linear(target_dim, num_components)
        self.sigma_head = nn.Linear(target_dim, num_components)
        # 🌟 仅在选择了 t 分布时，才挂载自由度预测头
        if self.dist_type == 't':
            self.nu_head = nn.Linear(target_dim, num_components)    

        # ==========================================
        # 🌟 优化：MDN 安全初始化策略
        # ==========================================
        # 1. 均值头：初始化为极小的值，防止初始预期收益率过高
        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        
        # 2. Sigma 头：让 softplus 初始化的结果在 1.0 左右
        # softplus(0.54) 大约等于 1.0
        nn.init.constant_(self.sigma_head.bias, 0.5413)
        
        if self.dist_type == 't':
            # 自由度头：初始化 nu 较大（例如 10），使其初始状态接近正态分布
            # 因为你的代码里有 + 2.01 的截断，softplus(x) + 2.01 = 10 -> x ≈ 2.07
            nn.init.constant_(self.nu_head.bias, 2.07)  

    def forward(self, x_macro, x_micro, x_micro_ind):
        """
        x_macro: (Batch, Macro_Dim) - 当前时刻的宏观因子 (2D张量)
        x_micro: (Batch, Micro_Dim) - 当前时刻的微观因子 (2D张量)
        x_micro_ind: (Batch,) - 行业整数索引 0~47 (Long Tensor)
        """
        # ------------------------------------------
        # 1. 宏观特征提取
        # ------------------------------------------
        h_macro = self.macro_net(x_macro) # 形状: (Batch, macro_hidden_dim)
        
        # 预测门控权重
        pi_logits = self.pi_head(h_macro)
        
        # ------------------------------------------
        # 2. 生成 FiLM 参数 (根据当前宏观环境)
        # ------------------------------------------
        gamma = self.film_gamma(h_macro) # 形状: (Batch, target_dim)
        beta = self.film_beta(h_macro)   # 形状: (Batch, target_dim)
        
        # ++++++++++++++++++++++++++++++++++++++++++
        # 3. 新增：Embedding 降维与特征拼接
        # ++++++++++++++++++++++++++++++++++++++++++
        # 将 [0, 1, ..., 47] 的整数索引，转化为 6 维稠密向量
        ind_emb = self.ind_embedding(x_micro_ind) # 形状: (Batch, 6)
        
        # 在特征维度 (dim=-1) 上拼接连续特征和行业 Embedding
        x_micro_combined = torch.cat([x_micro, ind_emb], dim=-1)
        
        # 修改：将拼接后的组合特征送入提取器
        h_micro = self.micro_extractor(x_micro_combined)
        
        # ------------------------------------------
        # 4. ★ 核心：执行 FiLM 调制 ★
        # ------------------------------------------
        # 宏观环境直接介入微观特征的表达。
        # (1.0 + gamma) 确保网络初始阶段 gamma 接近 0 时，维持原特征。
        h_modulated = (1.0 + gamma) * h_micro + beta
        
        # ------------------------------------------
        # 5. 专家网络最终预测
        # ------------------------------------------
        # mu = self.mu_head(h_modulated)
        # Mu 路径：直接使用纯粹的、未受宏观污染的微观特征
        mu = self.mu_head(h_micro)
        sigma = nn.functional.softplus(self.sigma_head(h_modulated)) + 1e-6
        
        # 3. 🌟 根据开关决定 Nu
        nu = None # 默认为空
        if self.dist_type == 't':
            # 保证自由度严格大于 2，以确保方差存在
            nu = nn.functional.softplus(self.nu_head(h_modulated)) + 2.01
            
        # 统一返回4个值，如果是 normal，nu 就是 None
        return pi_logits, mu, sigma, nu

# ==========================================
# 2. 模型定义：结构化 MDN (Mixture Density Network) + GRU + FiLM 调制
# ==========================================
class AdvancedStructuredMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, seq_length=12, micro_hidden_dim=128, macro_hidden_dim=64, num_components=3, dropout_rate=0.2):
        super(AdvancedStructuredMDN, self).__init__()
        self.num_components = num_components
        self.seq_length = seq_length
        
        # ==========================================
        # 通道 A: Macro Network (GRU 时序记忆网络)
        # 输入形状: (Batch, Seq_Length, Macro_Dim)
        # ==========================================
        # 单层 GRU 提取宏观时间序列的趋势特征
        self.macro_gru = nn.GRU(
            input_size=macro_dim, 
            hidden_size=macro_hidden_dim, 
            num_layers=1, 
            batch_first=True
        )
        
        # 对 GRU 的隐状态进行规范化
        self.macro_post_gru = nn.Sequential(
            nn.LayerNorm(macro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate)
        )
        
        # 门控预测头：基于宏观趋势预测机制概率 pi
        self.pi_head = nn.Linear(macro_hidden_dim, num_components)
        
        # ==========================================
        # ★ FiLM 调制发生器 (Modulation Generators) ★
        # ==========================================
        # 宏观网络生成缩放因子 Gamma 和平移因子 Beta
        # 注意：输出的维度必须和微观网络的深层特征维度一致 (micro_hidden_dim // 2)
        target_dim = micro_hidden_dim // 2
        self.film_gamma = nn.Linear(macro_hidden_dim, target_dim)
        self.film_beta = nn.Linear(macro_hidden_dim, target_dim)
        
        # ==========================================
        # 通道 B: Micro Network (专家网络)
        # 输入形状: (Batch, Micro_Dim)
        # ==========================================
        self.micro_extractor = nn.Sequential(
            nn.Linear(micro_dim, micro_hidden_dim),
            nn.LayerNorm(micro_hidden_dim), # 使用 LayerNorm 避免截面前视偏差
            nn.ELU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(micro_hidden_dim, target_dim),
            nn.LayerNorm(target_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate)
        )
        
        # 预测头 (Heads): 输入的是被 FiLM 调制后的特征
        self.mu_head = nn.Linear(target_dim, num_components)
        self.sigma_head = nn.Linear(target_dim, num_components)

    def forward(self, x_macro_seq, x_micro):
        """
        x_macro_seq: (Batch, Seq_Length, Macro_Dim) - 过去 seq_length 个月的宏观因子
        x_micro: (Batch, Micro_Dim) - 当前时刻的横截面微观因子
        """
        # ------------------------------------------
        # 1. 宏观时序处理
        # ------------------------------------------
        # GRU 返回 (output, hidden_state)
        # h_n 形状是 (num_layers, Batch, hidden_size)
        _, h_n = self.macro_gru(x_macro_seq) 
        
        # 提取最后一层的隐状态作为当前时刻的宏观表示
        h_macro = h_n[-1, :, :] # 形状: (Batch, macro_hidden_dim)
        h_macro = self.macro_post_gru(h_macro)
        
        # 预测门控权重
        pi_logits = self.pi_head(h_macro)
        
        # ------------------------------------------
        # 2. 生成 FiLM 参数
        # ------------------------------------------
        gamma = self.film_gamma(h_macro) # 形状: (Batch, target_dim)
        beta = self.film_beta(h_macro)   # 形状: (Batch, target_dim)
        
        # ------------------------------------------
        # 3. 微观特征提取
        # ------------------------------------------
        h_micro = self.micro_extractor(x_micro) # 形状: (Batch, target_dim)
        
        # ------------------------------------------
        # 4. ★ 核心：执行 FiLM 调制 ★
        # ------------------------------------------
        # 宏观的 Gamma 和 Beta 直接重塑微观特征的分布
        # 广播机制(Broadcasting)会自动处理 Batch 维度
        h_modulated = (1.0 + gamma) * h_micro + beta
        
        # ------------------------------------------
        # 5. 专家网络最终预测
        # ------------------------------------------
        mu = self.mu_head(h_modulated)
        # Sigma 必须严格为正
        sigma = nn.functional.softplus(self.sigma_head(h_modulated)) + 1e-6
        
        return pi_logits, mu, sigma
    

# ==========================================
# 4. 稳健的 NLL 损失函数 (支持时间衰减权重)
# ==========================================
def mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, target, weights, dist_type='normal'):
    """
    weights: shape (batch_size, 1)，距离现在越近权重大
    """
    log_pi = torch.log_softmax(pi_logits, dim=-1)

# 🌟 核心开关：底层分布切换
    if dist_type == 't':
        dist = torch.distributions.StudentT(df=nu, loc=mu, scale=sigma)
    else:
        dist = torch.distributions.Normal(mu, sigma)
    
    # 分布的 log_prob
    log_dist = dist.log_prob(target.expand_as(mu))
    log_mix = log_pi + log_dist
    
    # 每个样本基础的 NLL loss
    loss = -torch.logsumexp(log_mix, dim=-1) 
    
    # 乘以时间衰减权重后求平均
    # 确保 weights 的 shape 与 loss 对齐
    weighted_loss = loss * weights.squeeze()
    
    return weighted_loss.mean()/ weights.sum()

# ==========================================
# 5. 极速版双通道训练循环 (All-in-VRAM 加速)
# ==========================================
def train_structured_mdn(model, 
                         X_train_macro, X_train_micro, X_train_industry, y_train, w_train, 
                         X_val_macro=None, X_val_micro=None, X_val_industry=None, y_val=None, w_val=None,
                         epochs=500, batch_size=1024, lr=1e-3, patience=25): # 注意 batch_size 变了！
    if quick_test == True:
        epochs = 2
        batch_size = 512
        patience = 1

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=8, min_lr=1e-6,
    )
    # 极度清爽，直接将传入的 Tensor 打包，不需要任何格式转换和设备转移！
    tr_dataset = TensorDataset(X_train_macro, X_train_micro, X_train_industry, y_train.unsqueeze(1), w_train.unsqueeze(1))
    va_dataset = TensorDataset(X_val_macro, X_val_micro, X_val_industry, y_val.unsqueeze(1), w_val.unsqueeze(1))
    
    # 因为数据已经在显卡里了，不需要复杂的 worker
    train_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(va_dataset, batch_size=batch_size, shuffle=False)
    
    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0
    
    # 🌟 新增：用于记录 Loss 历史
    train_loss_history = []
    val_loss_history = []
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        # 注意看：这里去掉了 .to(device)！因为数据切片本来就在显存里了！
        for b_macro, b_micro, b_industry, b_y, b_w in train_loader:
            optimizer.zero_grad()
            
            # pi_logits, mu, sigma = model(b_macro, b_micro)

            pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)

            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * b_macro.size(0)
            
        train_loss /= len(train_loader.dataset)
        train_loss_history.append(train_loss)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for b_macro, b_micro, b_industry, b_y, b_w in val_loader:
                pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)
                loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)
                val_loss += loss.item() * b_macro.size(0)
        val_loss /= len(val_loader.dataset)
        val_loss_history.append(val_loss)

        scheduler.step(val_loss)
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break
                
    model.load_state_dict(best_model_state)
    return model, best_val_loss, train_loss_history, val_loss_history

# ==========================================
# 6. 启动模型训练流水线
# ==========================================
def calculate_time_decay_weights(df_dates, half_life_months=36):
    """
    计算基于半衰期的指数衰减权重。
    参数：
    df_dates: pd.Series, 包含 mthcaldt 日期
    half_life_months: 半衰期(月)。默认 36 个月(3年)权重减半。
    """
    max_date = df_dates.max()
    # 计算每个样本距离该窗口最新日期的月数差
    months_diff = (max_date.year - df_dates.dt.year) * 12 + (max_date.month - df_dates.dt.month)
    
    # 衰减因子 lambda: (0.5)^(1/half_life)
    decay_factor = np.power(0.5, 1.0 / half_life_months)
    
    # 计算基础权重
    weights = np.power(decay_factor, months_diff).values
    
    # 【极度关键】归一化权重，使得权重的均值为 1。
    # 如果不归一化，整个 Loss 的绝对值会变小，导致梯度的步长变相缩小！
    weights = weights / weights.mean() 
    return weights

In [13]:
if __name__ == "__main__":
    seed_everything(seed=random_seed)
    print("\n>>> [2/3] Initializing Data Window Generator and Global VRAM Tensors...")
        
    # 1. 实例化数据窗口生成器 (由于我们要用 Index 切片，重置一下 df 的索引非常重要)
    processed_df = processed_df.reset_index(drop=True)
    generator = DataWindowGenerator(df=processed_df, date_col='mthcaldt')
    window_gen = generator.expanding_window_generator(initial_train_years=initial_train_years, val_years=val_years, test_years=test_years)

    # ==============================================================
    # 🌟 终极黑客技巧：全局数据一次性打入 GPU (Global All-in-VRAM)
    # ==============================================================
    print("    -> Pushing the entire 30-year dataset into GPU VRAM at once...")
    global_X_mac = torch.FloatTensor(processed_df[macro_tensor_cols].values).to(device)
    global_X_mic = torch.FloatTensor(processed_df[micro_tensor_cols].values).to(device)
    global_ind = torch.LongTensor(processed_df['ffi49'].values).to(device) 
    global_Y     = torch.FloatTensor(processed_df['target_ret_final'].values).to(device)
    # ==============================================================

    history_logs = []
    all_oos_predictions = []
    macro_dim = len(macro_tensor_cols)
    micro_dim = len(micro_tensor_cols)
    # global_model = StructuredMDN(macro_dim, micro_dim, num_components=best_k).to(device)
    global_model = StructuredMDN_with_FiLM(macro_dim, micro_dim, num_industries=50, ind_emb_dim=6, num_components=best_k, dist_type=GLOBAL_DIST_TYPE).to(device)

    print("\n>>> [3/3] Starting Expanding Window Structured MDN Training (Zero PCIe Overhead)...")
    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(f"\n[Window {window_idx + 1}] Train: {info['train'][0]}~{info['train'][1]} | Val: {info['val'][0]}~{info['val'][1]} | Test: {info['test'][0]}~{info['test'][1]}")
        # --- 记录开始时间 ---
        start_time = time.time()

        # 提取各个集合在全局大表中的行索引 (Index)
        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        # 先拿到 train 月份的唯一宏观值（时间等权）
        train_months = train_df['mthcaldt'].unique()
        mac_for_norm = macro_df[macro_df['date'].isin(train_months)][macro_tensor_cols]
        mac_mean = torch.tensor(mac_for_norm.mean().values, dtype=torch.float32, device=device)
        mac_std  = torch.tensor(mac_for_norm.std().values,  dtype=torch.float32, device=device).clamp(min=1e-8)

        # 再应用到 merged 的 global tensor
        X_tr_mac = (global_X_mac[idx_tr] - mac_mean) / mac_std
        X_va_mac = (global_X_mac[idx_va] - mac_mean) / mac_std
        X_te_mac = (global_X_mac[idx_te] - mac_mean) / mac_std

        X_tr_mic, X_tr_ind, Y_tr = global_X_mic[idx_tr], global_ind[idx_tr], global_Y[idx_tr]
        X_va_mic, X_va_ind, Y_va = global_X_mic[idx_va], global_ind[idx_va], global_Y[idx_va]
        X_te_mic, X_te_ind       = global_X_mic[idx_te], global_ind[idx_te]
        
        # --- 唯一需要传输的是这一轮的动态权重 (体积极小，几乎0耗时) ---
        w_train_np = calculate_time_decay_weights(train_df['mthcaldt'], half_life_months=36)
        w_val_np   = calculate_time_decay_weights(val_df['mthcaldt'], half_life_months=12)
        w_train = torch.tensor(w_train_np, dtype=torch.float32, device=device)
        w_val   = torch.tensor(w_val_np,   dtype=torch.float32, device=device)
        
        # 动态学习率
        current_lr = 1e-3 if window_idx == 0 else 3e-4
        
        # 训练模型 (此时传入的全是已经在显存里的 Tensor，速度起飞)
        global_model, best_v_loss, tr_hist, va_hist = train_structured_mdn(
            model=global_model,     
            X_train_macro=X_tr_mac, X_train_micro=X_tr_mic, X_train_industry=X_tr_ind, y_train=Y_tr, w_train=w_train,
            X_val_macro=X_va_mac, X_val_micro=X_va_mic, X_val_industry=X_va_ind, y_val=Y_va, w_val=w_val,
            lr=current_lr, epochs=epochs, patience=patience, batch_size=batch_size
        )

        # ----------------------------------------------------
        # 基于 Train + Val 联合集合进行 Fine-tune (防止灾难性遗忘)
        # ----------------------------------------------------
        best_epoch = len(tr_hist)
        finetune_epochs = max(5, best_epoch // 10)  # 只用10%的epoch做微调
        print(f"  Phase 1 done: best_epoch={best_epoch}, finetune_epochs={finetune_epochs}")
        
        # 🌟 修复：合并 Train 和 Val 的 Tensor，保持历史记忆的同时吃进最新数据
        X_tr_mac_ft = torch.cat([X_tr_mac, X_va_mac], dim=0)
        X_tr_mic_ft = torch.cat([X_tr_mic, X_va_mic], dim=0)
        X_tr_ind_ft = torch.cat([X_tr_ind, X_va_ind], dim=0)
        Y_tr_ft     = torch.cat([Y_tr, Y_va], dim=0)
        
        # 重新计算合并后的时间衰减权重 (严谨做法)
        train_val_df = pd.concat([train_df, val_df])
        w_ft_np = calculate_time_decay_weights(train_val_df['mthcaldt'], half_life_months=36)
        w_ft = torch.FloatTensor(w_ft_np).to(device)

        global_model, _, _, _ = train_structured_mdn(
            model=global_model,
            X_train_macro=X_tr_mac_ft, X_train_micro=X_tr_mic_ft, X_train_industry=X_tr_ind_ft, # 使用联合数据
            y_train=Y_tr_ft,           w_train=w_ft,
            X_val_macro=X_va_mac,      X_val_micro=X_va_mic,      X_val_industry=X_va_ind,      # 仅作占位，无需关心
            y_val=Y_va,                w_val=w_val,
            lr=current_lr * 0.1,       # 学习率给 0.1 即可，0.05 也行，平滑过渡
            epochs=finetune_epochs,
            patience=finetune_epochs + 1  # 强制跑完，禁用 early stop
        )
        print(f"  Phase 2 fine-tune done ({finetune_epochs} epochs on Train+Val)")


        # --- 记录结束时间 ---
        end_time = time.time()
        duration = end_time - start_time
        # --- 记录指标 ---
        history_logs.append({
            'window': window_idx + 1,
            'train_period': f"{info['train'][0]} to {info['train'][1]}",
            'test_period': f"{info['test'][0]} to {info['test'][1]}",
            'duration_sec': duration,
            'best_val_loss': best_v_loss,
            'epochs_run': len(tr_hist),
            'train_loss_trace': tr_hist, # 存储为一个 list，方便后续绘图
            'val_loss_trace': va_hist
        })
        
        # --- 样本外预测 ---
        global_model.eval()
        with torch.no_grad():
            pi_logits, mu, sigma, nu = global_model(X_te_mac, X_te_mic, X_te_ind)
            pi = torch.softmax(pi_logits, dim=-1)

        # --- 结果回传至 CPU 保存 ---
        preds_df = test_df[['permno', 'mthcaldt', 'target_ret_final']].copy()
        preds_df['pi_vec'] = [json.dumps(vec.tolist()) for vec in pi.cpu().numpy()]
        preds_df['pi_logits_vec'] = [json.dumps(vec.tolist()) for vec in pi_logits.cpu().numpy()]
        preds_df['mu_vec'] = [json.dumps(vec.tolist()) for vec in mu.cpu().numpy()]
        preds_df['sigma_vec'] = [json.dumps(vec.tolist()) for vec in sigma.cpu().numpy()]
        
        # 🌟 无论什么分布，都强制生成 nu_vec 这一列，保持 DataFrame 结构永远一致
        if nu is not None:
            preds_df['nu_vec'] = [json.dumps(vec.tolist()) for vec in nu.cpu().numpy()]
        else:
            preds_df['nu_vec'] = None  # 没有自由度 (比如正态分布)，直接整列填 None
        
        all_oos_predictions.append(preds_df)
        
        # 测试用，仅跑几个窗口
        if quick_test == True and window_idx == 1: break

    print("\n>>> Pipeline complete! Aggregating all Out-of-Sample predictions...")
    if len(all_oos_predictions) > 0:
        final_oos_df = pd.concat(all_oos_predictions, ignore_index=True)
        metrics_df = pd.DataFrame(history_logs)
        print(f"Successfully generated {len(final_oos_df)} predictions.")

Global seed set to 42 to ensure reproducibility.

>>> [2/3] Initializing Data Window Generator and Global VRAM Tensors...
    -> Pushing the entire 30-year dataset into GPU VRAM at once...

>>> [3/3] Starting Expanding Window Structured MDN Training (Zero PCIe Overhead)...

[Window 1] Train: 1991-01~2005-12 | Val: 2006-01~2006-12 | Test: 2007-01~2007-12
  Phase 1 done: best_epoch=2, finetune_epochs=5
  Phase 2 fine-tune done (5 epochs on Train+Val)

[Window 2] Train: 1991-01~2006-12 | Val: 2007-01~2007-12 | Test: 2008-01~2008-12
  Phase 1 done: best_epoch=2, finetune_epochs=5
  Phase 2 fine-tune done (5 epochs on Train+Val)

>>> Pipeline complete! Aggregating all Out-of-Sample predictions...
Successfully generated 84807 predictions.


In [14]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
import random
import time
from datetime import datetime
warnings.filterwarnings('ignore')
from scipy.stats import t      # 🌟 换成 t 分布
from scipy.optimize import brentq
from tqdm import tqdm
from scipy.stats import norm

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

# 如果没有final_oos_df, 则从parquet加载
if 'final_oos_df' not in locals():
    print(">>> Loading final_oos_df from Parquet...")
    final_oos_df = pd.read_parquet('/Users/jianbinchen/Downloads/final_oos_df_20260430_0833.parquet') # local 路径
    print(f"final_oos_df loaded with shape: {final_oos_df.shape}")

# =================================================================
# 1. 核心数学工具：混合高斯分布的 CDF 和 ES 计算
# =================================================================

def mdn_cdf(y, pi, mu, sigma, nu=None):
    """
    计算混合高斯分布在 y 处的累积概率 P(Y <= y)
    公式: F(y) = sum( pi_k * Phi((y - mu_k) / sigma_k) )
    """
    """
    兼容 N 和 t 的混合分布累积概率
    """
    if nu is None:
        # norm.cdf 是标准正态分布的累积函数
        return np.sum(pi * norm.cdf((y - mu) / sigma))
    else:
        # scipy.stats.t.cdf 的 loc(均值) 默认0，scale(标准差) 默认1，需要标准化 y
        return np.sum(pi * t.cdf((y - mu) / sigma, df=nu))


def mdn_var_objective(y, pi, mu, sigma, nu, alpha):
    """
    求解 VaR 的目标函数：F(y) - alpha = 0
    """
    return mdn_cdf(y, pi, mu, sigma, nu) - alpha

# def calculate_es_analytical(alpha, pi, mu, sigma, nu=None):
#     """
#     混合高斯分布的 Expected Shortfall (ES) 解析解
#     公式参考: ES_alpha = -(1/alpha) * sum( pi_k * mu_k * Phi(d1) - sigma_k * phi(d1) )
#     其中 d1 是标准化后的 VaR 阈值
#     """
#     # 1. 首先通过数值求根找到 VaR (使得累积概率等于 alpha)
#     # 搜索区间设定在均值的上下 10 倍标准差之间，确保覆盖尾部
#     try:
#         lower_bound = np.min(mu - 10 * sigma)
#         upper_bound = np.max(mu + 10 * sigma)
#         var_alpha = brentq(mdn_var_objective, lower_bound, upper_bound, args=(pi, mu, sigma, alpha))
#     except:
#         # 如果求根失败（极罕见），使用加权均值作为退路
#         print(">>> Warning: Root finding for VaR failed. Using weighted mean as fallback.")
#         return np.sum(pi * mu), np.sum(pi * mu)

#     # 2. 计算每个组件在 VaR 处的贡献
#     # d = (VaR - mu) / sigma
#     d = (var_alpha - mu) / sigma
    
#     if nu is None:
#         # 正态分布解析积分
#         term1 = mu * norm.cdf(d)
#         term2 = sigma * norm.pdf(d)
#     else:
#         # t 分布解析积分
#         term1 = mu * t.cdf(d, df=nu)
#         # 积分核：sigma * [(nu + d^2) / (nu - 1)] * t_pdf(d)
#         term2 = sigma * ((nu + d**2) / (nu - 1)) * t.pdf(d, df=nu)
    
#     # ES = -(1/alpha) * Integral from -inf to VaR of (y * f(y)) dy
#     # 对于混合高斯，积分具有解析形式
#     es_val = -(1/alpha) * np.sum(pi * (term1 - term2))
    
#     return var_alpha, es_val

def calculate_es_analytical(alpha, pi, mu, sigma, nu=None):
    """
    混合分布的 Expected Shortfall (ES) 解析解 (自动路由 Normal 或 t-Distribution)
    加入动态边界扩张，完美解决 t 分布极度厚尾导致的求根失败问题。
    """
    # ==========================================
    # 🌟 修复核心：动态扩张求根区间
    # ==========================================
    # 1. 设定一个宽泛的初始猜测区间
    lower_bound = np.min(mu) - 10 * np.max(sigma)
    upper_bound = np.max(mu) + 10 * np.max(sigma)
    
    # 2. 动态向外扩张，直到找到一正一负的区间 (brentq的硬性要求)
    # f_lower 必须 < 0 (即 CDF < alpha)，f_upper 必须 > 0
    step = np.max(sigma) * 5 + 0.1 # 扩张步长
    
    for _ in range(100):  # 最多外扩 100 次，防止死循环
        val_lower = mdn_var_objective(lower_bound, pi, mu, sigma, nu, alpha)
        val_upper = mdn_var_objective(upper_bound, pi, mu, sigma, nu, alpha)
        
        if val_lower < 0 and val_upper > 0:
            break  # 成功包围住 VaR，跳出循环！
            
        if val_lower >= 0:
            lower_bound -= step  # 左侧不够极度，向左继续扩张
        if val_upper <= 0:
            upper_bound += step  # 右侧不够极度，向右继续扩张
            
        step *= 1.5  # 指数级加速扩张，对付极度厚尾
        
    # 3. 使用安全的区间进行高精度求根
    try:
        var_alpha = brentq(mdn_var_objective, lower_bound, upper_bound, args=(pi, mu, sigma, nu, alpha))
    except Exception as e:
        # 如果极极端情况下依然失败，安全退回到均值
        print(f">>> Warning: Root finding failed safely. {e}")
        return np.sum(pi * mu), np.sum(pi * mu)

    # ==========================================
    # 积分项计算 (保持不变)
    # ==========================================
    d = (var_alpha - mu) / sigma
    
    if nu is None:
        # 正态分布解析积分
        term1 = mu * norm.cdf(d)
        term2 = sigma * norm.pdf(d)
    else:
        # t 分布解析积分
        term1 = mu * t.cdf(d, df=nu)
        # 积分核：sigma * [(nu + d^2) / (nu - 1)] * t_pdf(d)
        term2 = sigma * ((nu + d**2) / (nu - 1)) * t.pdf(d, df=nu)
        
    es_val = -(1/alpha) * np.sum(pi * (term1 - term2))
    
    return var_alpha, es_val

def calculate_crps_numerical(y_true, pi, mu, sigma, nu=None):
    """
    通过数值积分计算混合高斯分布的 CRPS
    """
    # 1. 构建积分网格 z
    # 因为我们的目标收益率已经用 VIX 缩放过，绝大多数值落在 [-5, 5] 之间。
    # 设置 [-10, 10] 的区间和 1000 个格点，足以保证极高的积分精度。
    z = np.linspace(-10, 10, 1000)
    
    # 2. 计算网格上每一个点的预测累积概率 F(z)
    cdf_z = np.zeros_like(z)
    for k in range(len(pi)):
        # 向量化计算：算出每一个网格点的正态 CDF 并加权
        if nu is None:
            cdf_z += pi[k] * norm.cdf((z - mu[k]) / sigma[k])
        else:
            cdf_z += pi[k] * t.cdf((z - mu[k]) / sigma[k], df=nu[k])

    # 3. 真实发生的阶跃函数 (Heaviside step function)
    # 当 z 小于真实收益率时为 0，大于等于时为 1
    step_z = (z >= y_true).astype(float)
    
    # 4. 计算平方差
    squared_diff = (cdf_z - step_z) ** 2
    
    # 5. 使用梯形法则 (Trapezoidal rule) 计算定积分
    crps_val = np.trapezoid(squared_diff, z)
    
    return crps_val

# =================================================================
# 2. 主程序：从 SQLite 读取数据并执行计算
# =================================================================

results = []

print(">>> Extracting Tail Risk Factors (ES & VaR)...")
# 使用 tqdm 显示进度
for idx, row in tqdm(final_oos_df.iterrows(), total=len(final_oos_df)):
    # 1. 解码 JSON 字符串回 Numpy 数组
    pi = np.array(json.loads(row['pi_vec']))
    mu = np.array(json.loads(row['mu_vec']))
    sigma = np.array(json.loads(row['sigma_vec']))
    y_true = row['target_ret_final'] # 真实发生的（缩放后的）收益率

    # 🌟 2. 安全读取自由度 nu (智能判断分布类型)
    nu = None
    if 'nu_vec' in row and pd.notna(row['nu_vec']):
        nu = np.array(json.loads(row['nu_vec']))
    
    # 2. 计算风险指标：VaR 和 ES
    # 计算 1% 分位数的 VaR 和 ES
    alpha = 0.01
    var_1, es_1 = calculate_es_analytical(alpha, pi, mu, sigma, nu)
    #计算 5% 分位数的 VaR 和 ES
    alpha = 0.05
    var_5, es_5 = calculate_es_analytical(alpha, pi, mu, sigma, nu)
    # 计算 10% 分位数的 VaR 和 ES
    alpha = 0.10
    var_10, es_10 = calculate_es_analytical(alpha, pi, mu, sigma, nu)
    # 计算 90% 分位数的 VaR 和 ES
    alpha = 0.90
    var_90, es_90 = calculate_es_analytical(alpha, pi, mu, sigma, nu)
    # 计算 95% 分位数的 VaR 和 ES
    alpha = 0.95
    var_95, es_95 = calculate_es_analytical(alpha, pi, mu, sigma, nu)
    # 计算 99% 分位数的 VaR 和 ES
    alpha = 0.99
    var_99, es_99 = calculate_es_analytical(alpha, pi, mu, sigma, nu)

    # 3. 计算 PIT (Probability Integral Transform)
    # 即：真实收益率 y_true 在预测分布中处于什么百分位
    pit_val = mdn_cdf(y_true, pi, mu, sigma, nu)

    # 新增：计算 CRPS
    crps_val = calculate_crps_numerical(y_true, pi, mu, sigma, nu)

    # 4. 计算分布的高阶矩 (用于控制变量)
    # 混合分布均值: sum(pi * mu)
    mean_pred = np.sum(pi * mu)

# 🌟 4. 高阶矩计算 (方差会随 t 分布自由度变化)
    mean_pred = np.sum(pi * mu)
    if nu is None:
        component_var = sigma**2
    else:
        # t分布的真实方差公式: sigma^2 * (nu / (nu - 2))
        # (我们的网络中 nu_head 加了 2.01，所以 nu 必然大于 2，不会出现除以0的错误)
        component_var = sigma**2 * (nu / (nu - 2))
    # 混合分布方差: sum(pi * (mu^2 + sigma^2)) - mean^2
    var_pred = np.sum(pi * (mu**2 + component_var)) - mean_pred**2
    
    results.append({
        'permno': row['permno'],
        'date': row['mthcaldt'],
        'es_1': es_1,
        'var_1': var_1,
        'es_5': es_5,       
        'var_5': var_5,     
        'es_10': es_10,     
        'var_10': var_10,
        'es_90': es_90,
        'var_90': var_90,
        'es_95': es_95,
        'var_95': var_95,
        'es_99': es_99,
        'var_99': var_99,     
        'pit': pit_val,     
        'crps': crps_val,   # <--- 把 CRPS 也存进最终的 DataFrame 里
        'mean_pred': mean_pred,
        'vol_pred': np.sqrt(var_pred),
        'realized_ret': y_true
    })

ES_df = pd.DataFrame(results)

>>> Extracting Tail Risk Factors (ES & VaR)...


100%|██████████| 84807/84807 [09:08<00:00, 154.67it/s]


In [15]:
# 指定保存路径
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

if env == 'kaggle':
    path = '/kaggle/working'
elif env == 'colab':
    # 1. 挂载 Google Drive
    google.colab.drive.mount('/content/drive')
    path = '/content/drive/MyDrive/Colab Notebooks/'
else:
    path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'

# 执行保存
final_oos_df.to_parquet(os.path.join(path, f'final_oos_df_{timestamp}.parquet'), index=False)
metrics_df.to_parquet(os.path.join(path, f'metrics_df_{timestamp}.parquet'), index=False)
ES_df.to_parquet(os.path.join(path, f'ES_df_{timestamp}.parquet'), index=False)
print(f"保存成功！文件位置: {path}")


MessageError: Failed to issue request POST https://colab.research.google.com/tun/m/credentials-propagation/gpu-t4-s-kkb-ass1a2-3jijt125fbusv?authtype=dfs_ephemeral&version=2&dryrun=false&propagate=true&record=false&authuser=0: Bad Request
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 400 (Bad Request)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/error_lo

In [16]:
if env == 'colab':
    # 运行完你的所有逻辑后，调用此命令会自动释放当前后端实例
    google.colab.runtime.unassign()